In [9]:
import gc
import random
import time

import torch

from edge_detector.backbones.baselines import create_baseline_backbone
from edge_detector.scripts.benchmark import (
    benchmark_module,
    configure_cpu,
    feature_shapes,
)


INPUT_SIZE = 640
THREADS = 4
WARMUP = 1
RUNS = 1
RANDOM_SEED = 42
COOLDOWN_SECONDS = 0

BACKBONES = [
    "mobilenetv4_conv_small_035",
    "mobilenetv4_conv_small_050",
    "mobilenetv4_conv_small",
    "shufflenet_v2_x0_5",
    "shufflenet_v2_x1_0",
    "shufflenet_v2_x1_5",
    "shufflenet_v2_x2_0",
    "fasternet_t0",
    "fasternet_t1",
    "fasternet_t2",
    "ghostnet_050",
    "ghostnet_100",
    "ghostnet_130",
    "yolov8n",
    "yolov8s",
    "mobilenetv3_large_100",
    "fbnetv3_d",
    "mobileone_s0",
    "repvit_m0_9",
    "starnet_s1",
    "yolov6n_efficientrep",
]

configure_cpu(THREADS)

In [4]:
x = torch.randn(
    1,
    3,
    INPUT_SIZE,
    INPUT_SIZE,
    dtype=torch.float32,
)

random.Random(RANDOM_SEED).shuffle(BACKBONES)
print(f"Models: {len(BACKBONES)}")

Models: 21


In [10]:
def benchmark_backbone(model_name: str):
    model = create_baseline_backbone(
        model_name=model_name,
        deploy=True,
    ).eval()

    shapes = feature_shapes(model, x)
    channels = tuple(shape[1] for shape in shapes)

    benchmark = benchmark_module(
        model,
        x,
        warmup=WARMUP,
        runs=RUNS,
    )

    result = {
        "model": model_name,
        "channels": channels,
        "features": shapes,
        **benchmark.as_dict(),
    }

    return result, benchmark

In [11]:
results = []

for index, model_name in enumerate(BACKBONES, start=1):
    result, benchmark = benchmark_backbone(model_name)
    results.append(result)

    c3, c4, c5 = result["features"]

    print(
        f"[{index:2d}/{len(BACKBONES)}] "
        f"{result['model']:30s} | "
        f"C3={str(c3):18s} "
        f"C4={str(c4):18s} "
        f"C5={str(c5):18s} | "
        f"{benchmark}"
    )

    gc.collect()

    if COOLDOWN_SECONDS and index < len(BACKBONES):
        time.sleep(COOLDOWN_SECONDS)

[ 1/21] mobilenetv4_conv_small_035     | C3=(1, 24, 80, 80)    C4=(1, 32, 40, 40)    C5=(1, 48, 20, 20)    | params=  0.175 M | median=  35.596 ms | mean=  35.596 ms | min=  35.596 ms | max=  35.596 ms
[ 2/21] mobilenetv4_conv_small_050     | C3=(1, 32, 80, 80)    C4=(1, 48, 40, 40)    C5=(1, 64, 20, 20)    | params=  0.309 M | median=  43.916 ms | mean=  43.916 ms | min=  43.916 ms | max=  43.916 ms
[ 3/21] mobilenetv4_conv_small         | C3=(1, 64, 80, 80)    C4=(1, 96, 40, 40)    C5=(1, 128, 20, 20)   | params=  1.137 M | median=  63.641 ms | mean=  63.641 ms | min=  63.641 ms | max=  63.641 ms
[ 4/21] shufflenet_v2_x0_5             | C3=(1, 48, 80, 80)    C4=(1, 96, 40, 40)    C5=(1, 192, 20, 20)   | params=  0.143 M | median= 126.906 ms | mean= 126.906 ms | min= 126.906 ms | max= 126.906 ms
[ 5/21] shufflenet_v2_x1_0             | C3=(1, 116, 80, 80)   C4=(1, 232, 40, 40)   C5=(1, 464, 20, 20)   | params=  0.776 M | median=  89.890 ms | mean=  89.890 ms | min=  89.890 ms | max=  

In [12]:
sorted_results = sorted(
    results,
    key=lambda result: result["median_ms"],
)

header = (
    f"{'Backbone':30s} "
    f"{'Channels C3/C4/C5':>20s} "
    f"{'Params, M':>10s} "
    f"{'Median, ms':>12s} "
    f"{'Mean, ms':>10s} "
    f"{'Min, ms':>10s} "
    f"{'Max, ms':>10s}"
)

print(header)
print("-" * len(header))

for result in sorted_results:
    channels = "/".join(map(str, result["channels"]))

    print(
        f"{result['model']:30s} "
        f"{channels:>20s} "
        f"{result['params'] / 1e6:10.3f} "
        f"{result['median_ms']:12.2f} "
        f"{result['mean_ms']:10.2f} "
        f"{result['min_ms']:10.2f} "
        f"{result['max_ms']:10.2f}"
    )

Backbone                          Channels C3/C4/C5  Params, M   Median, ms   Mean, ms    Min, ms    Max, ms
------------------------------------------------------------------------------------------------------------
mobilenetv4_conv_small_035                 24/32/48      0.175        35.60      35.60      35.60      35.60
mobilenetv4_conv_small_050                 32/48/64      0.309        43.92      43.92      43.92      43.92
mobilenetv4_conv_small                    64/96/128      1.137        63.64      63.64      63.64      63.64
yolov8n                                  64/128/256      1.273        69.18      69.18      69.18      69.18
shufflenet_v2_x1_0                      116/232/464      0.776        89.89      89.89      89.89      89.89
yolov6n_efficientrep                     64/128/256      3.133        97.49      97.49      97.49      97.49
shufflenet_v2_x0_5                        48/96/192      0.143       126.91     126.91     126.91     126.91
fasternet_t0       